# Install & Imports

In [1]:
# Install necessary libraries if you haven't (Run this once)
%pip install torch torchvision matplotlib scikit-learn

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/113.8 MB ? eta -:--:--
   ---------------------------------------- 0.5/113.8 MB 989.2 kB/s eta 0:01:55
   ---------------------------------------- 0.8/113.8 MB 987.4 kB/s eta 0:01:55
   ---------------------------------------- 1.0/113.8 MB 1.1 MB/s eta 0:01:46
   ---------------------------------------- 1.0/113.8 MB 1.1 MB/s eta 0:01:46
   ---------------------------------------- 1.3/113.8 MB 945.5 kB/s eta 0:01:59
    --------------------------------------- 1.6/113.8 MB 999.0 kB/s eta 0:01:53
    --------------------------------------- 1.8/113.8 MB 1.1 MB/s eta 0:01:43
    -----------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# The Setup (Imports & GPU Check)

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import os

# 1. Setup Device (Uses GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Training on: {device}")

# 2. Define our specific aesthetic classes
CLASSES = ["goth", "gym_rat", "old_money", "streetwear", "y2k"]
print(f"🎯 Target Classes: {CLASSES}")

🚀 Training on: cpu
🎯 Target Classes: ['goth', 'gym_rat', 'old_money', 'streetwear', 'y2k']


# The Data Pipeline (Crucial Step)

In [3]:
# Define standard stats for ResNet (Don't change these numbers)
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),       # ResNet expects 224x224
        transforms.RandomHorizontalFlip(),   # Augmentation (Gym)
        transforms.RandomRotation(15),       # Augmentation (Gym)
        transforms.ToTensor(),
        transforms.Normalize(mean, std)      # Math magic
    ])
}

# Load the Data
data_dir = 'dataset' # Assuming you are running this from inside the 'training' folder
image_datasets = datasets.ImageFolder(data_dir, data_transforms['train'])
dataloaders = torch.utils.data.DataLoader(image_datasets, batch_size=8, shuffle=True)

print(f"✅ Loaded {len(image_datasets)} images across {len(image_datasets.classes)} classes.")

✅ Loaded 297 images across 5 classes.


# The Brain Surgery (Transfer Learning)

In [4]:
# 1. Download the Genius (Pre-trained ResNet18)
model = models.resnet18(weights='IMAGENET1K_V1')

# 2. FREEZE the Genius (Lock the early layers)
# We tell the optimizer: "Don't touch these weights, they are already perfect."
for param in model.parameters():
    param.requires_grad = False

# 3. TRANSPLANT the Head
# ResNet's final layer is called 'fc'. It has 512 inputs.
# We replace it with a new layer that has 5 outputs (our classes).
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(CLASSES))

# Move model to GPU/CPU
model = model.to(device)

print("🧠 Model surgery complete. Ready to learn Drip.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\nawaz/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:48<00:00, 966kB/s]


🧠 Model surgery complete. Ready to learn Drip.


# The Training Loop

In [5]:
# Define "Judge" (Loss) and "Coach" (Optimizer)
criterion = nn.CrossEntropyLoss()

# NOTICE: We only optimize 'model.fc.parameters()' (The new head). 
# The rest of the model is frozen and won't cost us computation time.
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# The Loop
num_epochs = 20

print("🔥 Starting Training...")

for epoch in range(num_epochs):
    model.train() # Set to training mode
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in dataloaders:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()           # Reset gradients
        outputs = model(inputs)         # Forward pass
        loss = criterion(outputs, labels) # Calculate Loss
        loss.backward()                 # Backward pass
        optimizer.step()                # Update weights
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {running_loss/len(dataloaders):.4f} | Acc: {epoch_acc:.2f}%")

print("✨ Training Complete!")

🔥 Starting Training...
Epoch 1/20 | Loss: 1.6010 | Acc: 29.63%
Epoch 2/20 | Loss: 1.4315 | Acc: 42.42%
Epoch 3/20 | Loss: 1.2883 | Acc: 47.47%
Epoch 4/20 | Loss: 1.1565 | Acc: 54.21%
Epoch 5/20 | Loss: 1.1430 | Acc: 56.57%
Epoch 6/20 | Loss: 1.0995 | Acc: 58.59%
Epoch 7/20 | Loss: 1.0388 | Acc: 59.93%
Epoch 8/20 | Loss: 1.0056 | Acc: 63.97%
Epoch 9/20 | Loss: 1.0372 | Acc: 59.93%
Epoch 10/20 | Loss: 0.9875 | Acc: 62.29%
Epoch 11/20 | Loss: 0.9005 | Acc: 64.65%
Epoch 12/20 | Loss: 0.8998 | Acc: 67.34%
Epoch 13/20 | Loss: 0.9283 | Acc: 69.36%
Epoch 14/20 | Loss: 0.8516 | Acc: 73.74%
Epoch 15/20 | Loss: 0.9011 | Acc: 68.01%
Epoch 16/20 | Loss: 0.8054 | Acc: 71.38%
Epoch 17/20 | Loss: 0.7266 | Acc: 72.73%
Epoch 18/20 | Loss: 0.7968 | Acc: 71.38%
Epoch 19/20 | Loss: 0.7721 | Acc: 72.39%
Epoch 20/20 | Loss: 0.7454 | Acc: 72.73%
✨ Training Complete!


# Save the Brain (Deployment)

In [6]:
# Define save path
save_path = os.path.join("..", "backend", "weights", "drip_model.pth")
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# Save the weights
torch.save(model.state_dict(), save_path)
print(f"💾 Model saved successfully at: {save_path}")

💾 Model saved successfully at: ..\backend\weights\drip_model.pth
